<a href="https://colab.research.google.com/github/kayokfds/trabalhoooo/blob/main/02_Dados_Spread.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dados estão no Google Drive, em FGV > TCC > Dados

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount("/content/drive")
import os
import requests
import zipfile
from io import BytesIO
from tqdm import tqdm
from datetime import datetime, timedelta

Mounted at /content/drive


In [ ]:
caminho_dados = "/content/drive/MyDrive/FGV/TCC/Dados/Brutos"
caminho_output = "/content/drive/MyDrive/FGV/TCC/Dados"

##Baixar os Dados

In [ ]:
# Gera TODAS as datas (inclusive fins de semana e feriados)
d1 = datetime.strptime('2016-01-01', '%Y-%m-%d')
d2 = datetime.strptime('2025-12-31', '%Y-%m-%d')
datas = [(d1 + timedelta(days=i)) for i in range((d2 - d1).days + 1)]

In [ ]:
import requests
import zipfile
import io


def baixar_taxaswap(data):
    """
    Baixa o arquivo TaxaSwap da B3 para uma determinada data.

    Parâmetros
    ----------
    data : str
        Data no formato 'YYYY-MM-DD'.

    Retorna
    -------
    pandas.DataFrame
        Arquivo TaxaSwap já carregado em memória.
    """

    # --------------------------------------------------------
    # 1. Montar nome do arquivo
    # --------------------------------------------------------

    data = pd.to_datetime(data)

    nome_arquivo = f"TS{data.strftime('%y%m%d')}.ex_"

    url = (
        "https://www.b3.com.br/pesquisapregao/download"
        f"?filelist={nome_arquivo}"
    )

    # --------------------------------------------------------
    # 2. Baixar arquivo da B3
    # --------------------------------------------------------

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "*/*",
        "Referer": "https://www.b3.com.br/"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=60
    )

    response.raise_for_status()

    if len(response.content) == 0:
        raise ValueError(
            f"A B3 retornou arquivo vazio para {data.date()}."
        )

    # --------------------------------------------------------
    # 3. Primeiro ZIP
    # --------------------------------------------------------

    zip_externo = zipfile.ZipFile(
        io.BytesIO(response.content)
    )

    # O ZIP externo contém o executável SFX
    sfx_bytes = zip_externo.read(
        zip_externo.namelist()[0]
    )

    # --------------------------------------------------------
    # 4. Segundo ZIP — dentro do SFX
    # --------------------------------------------------------

    zip_interno = zipfile.ZipFile(
        io.BytesIO(sfx_bytes)
    )

    candidatos = [
        nome
        for nome in zip_interno.namelist()
        if "taxaswap" in nome.lower()
    ]

    if not candidatos:
        raise FileNotFoundError(
            f"TaxaSwap.txt não encontrado para {data.date()}."
        )

    # --------------------------------------------------------
    # 5. Ler TaxaSwap.txt
    # --------------------------------------------------------

    taxaswap_bytes = zip_interno.read(candidatos[0])

    texto = taxaswap_bytes.decode("latin-1")

    # --------------------------------------------------------
    # 6. Retornar o arquivo bruto
    # --------------------------------------------------------

    return texto

In [ ]:
linhas_saida = []
import time

for i, data in enumerate(datas):
    try:
        texto = baixar_taxaswap(data.strftime('%Y-%m-%d'))
    except Exception:
        continue

    for l in texto.split("\n"):
        if len(l) < 66:
            continue
        if l[26:32] == "DIxPRE" and l[33:40].strip() == "":
            dc = int(l[41:46])
            taxa = int(l[52:66]) / 1e7
            linhas_saida.append({"data": data.strftime("%Y-%m-%d"), "dias_corridos": dc, "taxa": taxa})

    if i % 100 == 0:
        print(f"{i}/{len(datas)} dias processados, {len(linhas_saida)} vertices coletados ate agora")

    time.sleep(0.3)

df = pd.DataFrame(linhas_saida)
df.head(10)

KeyboardInterrupt: 

##Exportar os Dados

In [ ]:
# 6. Salva CSV (sobrescreve sempre)
df.to_csv(f"{caminho_output}/dados_term_spread.csv")

##Datas com problema - Feito depois

In [ ]:
datas_sem_dado = pd.to_datetime([
    '2018-02-14', '2018-02-16', '2018-02-19',
    '2018-05-25', '2021-12-06', '2021-12-07',
    '2021-12-09', '2021-12-13', '2021-12-14',
    '2021-12-15', '2021-12-16', '2021-12-17',
    '2021-12-20', '2022-01-03', '2022-03-03',
    '2025-01-02', '2025-01-03', '2025-01-06',
    '2025-01-07', '2025-01-08', '2025-01-09',
    '2025-01-10', '2025-01-13', '2025-01-14',
    '2025-01-15', '2025-01-16', '2025-01-17',
    '2025-01-27', '2025-01-28', '2025-01-29',
    '2025-01-31', '2025-02-03', '2025-02-04',
    '2025-02-10', '2025-02-12', '2025-02-17',
    '2025-03-05', '2025-03-06', '2025-03-07',
    '2025-03-10', '2025-03-11', '2025-03-17',
    '2025-03-18', '2025-03-19', '2025-03-21',
    '2025-03-24', '2025-03-31', '2025-04-01',
    '2025-04-02', '2025-04-03', '2025-04-22',
    '2025-04-28', '2025-04-29', '2025-04-30',
    '2025-05-06', '2025-05-08', '2025-05-09',
    '2025-05-12', '2025-05-13', '2025-05-14',
    '2025-05-15', '2025-05-16', '2025-05-20',
    '2025-05-22', '2025-05-23',
])

linhas_recuperadas = []
ainda_faltando = []

for data in datas_sem_dado:
    try:
        texto = baixar_taxaswap(data.strftime('%Y-%m-%d'))
    except Exception as e:
        print(data.date(), "FALHOU:", e)
        ainda_faltando.append(data)
        continue

    encontrou_vertice = False
    for l in texto.split("\n"):
        if len(l) < 66:
            continue
        if l[26:32] == "DIxPRE" and l[33:40].strip() == "":
            dc = int(l[41:46])
            taxa = int(l[52:66]) / 1e7
            if 0 < taxa < 100:
                linhas_recuperadas.append({"data": data.strftime("%Y-%m-%d"), "dias_corridos": dc, "taxa": taxa})
                encontrou_vertice = True

    if encontrou_vertice:
        print(data.date(), "RECUPERADO")
    else:
        print(data.date(), "baixou mas sem vertices DIxPRE (gap real)")
        ainda_faltando.append(data)

print()
print(f"Recuperados: {len(datas_sem_dado) - len(ainda_faltando)} de {len(datas_sem_dado)}")
print(f"Ainda faltando: {len(ainda_faltando)}")
for d in ainda_faltando:
    print(" -", d.date())

2018-02-14 RECUPERADO
2018-02-16 RECUPERADO
2018-02-19 RECUPERADO
2018-05-25 baixou mas sem vertices DIxPRE (gap real)
2021-12-06 RECUPERADO
2021-12-07 RECUPERADO
2021-12-09 RECUPERADO
2021-12-13 RECUPERADO
2021-12-14 RECUPERADO
2021-12-15 RECUPERADO
2021-12-16 RECUPERADO
2021-12-17 RECUPERADO
2021-12-20 RECUPERADO
2022-01-03 RECUPERADO
2022-03-03 RECUPERADO
2025-01-02 RECUPERADO
2025-01-03 RECUPERADO
2025-01-06 RECUPERADO
2025-01-07 RECUPERADO
2025-01-08 RECUPERADO
2025-01-09 RECUPERADO
2025-01-10 RECUPERADO
2025-01-13 RECUPERADO
2025-01-14 RECUPERADO
2025-01-15 RECUPERADO
2025-01-16 RECUPERADO
2025-01-17 RECUPERADO
2025-01-27 RECUPERADO
2025-01-28 RECUPERADO
2025-01-29 RECUPERADO
2025-01-31 RECUPERADO
2025-02-03 RECUPERADO
2025-02-04 RECUPERADO
2025-02-10 RECUPERADO
2025-02-12 RECUPERADO
2025-02-17 RECUPERADO
2025-03-05 RECUPERADO
2025-03-06 RECUPERADO
2025-03-07 RECUPERADO
2025-03-10 RECUPERADO
2025-03-11 RECUPERADO
2025-03-17 RECUPERADO
2025-03-18 RECUPERADO
2025-03-19 RECUPERADO
2

In [ ]:
df_recuperado = pd.DataFrame(linhas_recuperadas)
df_recuperado.head()

,data,dias_corridos,taxa
0,2018-02-14,1,6.640
1,2018-02-14,6,6.643
2,2018-02-14,8,6.644
3,2018-02-14,13,6.644
4,2018-02-14,14,6.644


In [ ]:
# # 6. Salva CSV (sobrescreve sempre)
df_recuperado.to_csv(f"{caminho_output}/dados_term_spread_complementar.csv")

##Rascunho

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime, timedelta
import time

DATA_INICIO, DATA_FIM = '2016-01-01', '2025-12-31'
d1 = datetime.strptime(DATA_INICIO, '%Y-%m-%d')
d2 = datetime.strptime(DATA_FIM, '%Y-%m-%d')
datas = [(d1 + timedelta(days=i)) for i in range((d2 - d1).days + 1)]

resultados = []

for data in tqdm(datas, desc="Baixando ANBIMA"):
    # Formato da URL: dd/mm/yyyy
    data_fmt = data.strftime('%d/%m/%Y')
    params = {'dt': data_fmt, 'idioma': 'pt'}

    try:
        # A ANBIMA aceita GET com parâmetro dt=dd/mm/yyyy
        url = f'https://www.anbima.com.br/informacoes/est-termo/CZ.asp'
        resp = requests.get(url, params=params, timeout=15)

        if resp.status_code != 200:
            # Dia sem dado (fim de semana/feriado)
            continue

        soup = BeautifulSoup(resp.text, 'html.parser')

        # A tabela de prefixados está em uma <table> com os vértices e taxas
        # Procuramos a tabela que contém "PREFIXADOS (CIRCULAR 3.361)"
        table = None
        for t in soup.find_all('table'):
            if 'PREFIXADOS' in str(t):
                table = t
                break

        if not table:
            continue

        # Extrair linhas da tabela (cada linha tem 2 vértices)
        vertices = {}
        for tr in table.find_all('tr')[2:]:  # pula cabeçalho
            tds = tr.find_all('td')
            if len(tds) < 4:
                continue

            # Cada linha tem: vertice1 | taxa1 | vertice2 | taxa2
            try:
                v1 = int(tds[0].text.strip())
                r1 = float(tds[1].text.replace('.', '').replace(',', '.'))
                vertices[v1] = r1
            except: pass

            try:
                v2 = int(tds[2].text.strip())
                r2 = float(tds[3].text.replace('.', '').replace(',', '.'))
                vertices[v2] = r2
            except: pass

        if not vertices:
            continue

        # Busca vértices mais próximos (tolerância pequena)
        def buscar(alvo, tol):
            for d in range(alvo-tol, alvo+tol+1):
                if d in vertices:
                    return vertices[d]
            return None

        # Vértices em dias úteis
        t2520, t504, t252, t126, t63 = (
            buscar(2520, 30), buscar(504, 15), buscar(252, 10),
            buscar(126, 7), buscar(63, 5)
        )

        s10y2y = t2520 - t504 if t2520 and t504 else np.nan
        s10y1y = t2520 - t252 if t2520 and t252 else np.nan
        s6m3m  = t126 - t63 if t126 and t63 else np.nan

        if not (np.isnan(s10y2y) and np.isnan(s10y1y) and np.isnan(s6m3m)):
            resultados.append({
                'data': data.strftime('%Y-%m-%d'),
                'spread_10y_2y': s10y2y,
                'spread_10y_1y': s10y1y,
                'spread_6m_3m': s6m3m
            })

        time.sleep(0.5)  # Respeitar a ANBIMA, evita bloqueio

    except Exception as e:
        # print(f"Erro em {data_fmt}: {e}")
        continue

# Salva CSV
if resultados:
    df = pd.DataFrame(resultados)
    df = df.sort_values('data')
    df.to_csv(f"{caminho_output}/dados_term_spread.csv", index=False)
    print(f"\n✅ Salvo: dados_term_spread ({len(df)} dias)")
else:
    print("\n❌ Nenhum dado foi baixado. Verifique internet.")

Baixando ANBIMA: 100%|██████████| 3653/3653 [01:27<00:00, 41.66it/s]


❌ Nenhum dado foi baixado. Verifique internet.


In [ ]:
!pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 12.8 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


###Teste GPT

In [ ]:
import requests
import zipfile
import io
import pandas as pd
from datetime import date

# ============================================================
# TESTE 1 — baixar um TaxaSwap histórico diretamente da B3
# ============================================================

# Vamos testar 12/12/2025, data para a qual temos evidência
# de que existe o arquivo TS251212.ex_
data_teste = date(2025, 12, 12)

nome_arquivo = f"TS{data_teste.strftime('%y%m%d')}.ex_"

url = f"https://www.b3.com.br/pesquisapregao/download?filelist={nome_arquivo}"

print("Arquivo:", nome_arquivo)
print("URL:", url)
print("\nBaixando...")

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "*/*",
    "Referer": "https://www.b3.com.br/"
}

response = requests.get(
    url,
    headers=headers,
    timeout=60
)

print("Status HTTP:", response.status_code)
print("Tamanho recebido:", len(response.content), "bytes")
print("Content-Type:", response.headers.get("Content-Type"))

# Verificações básicas
if response.status_code != 200:
    raise Exception(
        f"Download falhou. Status HTTP: {response.status_code}"
    )

if len(response.content) == 0:
    raise Exception("A B3 retornou um arquivo vazio.")

print("\nPrimeiros 20 bytes:")
print(response.content[:20])

Arquivo: TS251212.ex_
URL: https://www.b3.com.br/pesquisapregao/download?filelist=TS251212.ex_

Baixando...
Status HTTP: 200
Tamanho recebido: 326450 bytes
Content-Type: application/octet-stream

Primeiros 20 bytes:
b'PK\x03\x04\x14\x00\x08\x08\x08\x00G\xb5\x19]\x00\x00\x00\x00\x00\x00'


In [ ]:
# ============================================================
# TESTE 2 — descompactar o arquivo
# ============================================================

arquivo = io.BytesIO(response.content)

print("É um ZIP válido?", zipfile.is_zipfile(arquivo))

if not zipfile.is_zipfile(arquivo):
    print("\nO conteúdo não é reconhecido diretamente como ZIP.")
    print("Vamos inspecionar o início do arquivo:")
    print(response.content[:500])
else:
    with zipfile.ZipFile(arquivo) as z:
        print("\nArquivos dentro do pacote:")
        for nome in z.namelist():
            print(" -", nome)

É um ZIP válido? True

Arquivos dentro do pacote:
 - TS251212.ex_


In [ ]:
# ============================================================
# TESTE 3 — abrir o TS251212.ex_ e inspecionar o conteúdo
# ============================================================

arquivo = io.BytesIO(response.content)

with zipfile.ZipFile(arquivo) as z:
    nome = z.namelist()[0]

    conteudo = z.read(nome)

print("Arquivo:", nome)
print("Tamanho:", len(conteudo), "bytes")

# Tentar identificar a codificação
for encoding in ["latin-1", "cp1252", "utf-8"]:
    try:
        texto = conteudo.decode(encoding)
        print(f"Encoding {encoding}: OK")
        break
    except UnicodeDecodeError:
        print(f"Encoding {encoding}: falhou")

print("\n" + "="*100)
print("PRIMEIRAS 20 LINHAS")
print("="*100)

linhas = texto.splitlines()

for i, linha in enumerate(linhas[:20], start=1):
    print(f"{i:02d}: {repr(linha)}")

print("\nNúmero total de linhas:", len(linhas))

Arquivo: TS251212.ex_
Tamanho: 468751 bytes
Encoding latin-1: OK

PRIMEIRAS 20 LINHAS
01: 'MZ¡\x00\x02\x00\x00\x00 \x00\x00\x00ÿÿ\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00@\x00\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x18\x03\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x

GPT 2

In [ ]:
import os
import subprocess
import shutil

# Salvar o arquivo baixado
arquivo_sfx = "/content/TS251212.ex_"

with open(arquivo_sfx, "wb") as f:
    f.write(response.content)

print("Arquivo salvo:", arquivo_sfx)
print("Tamanho:", os.path.getsize(arquivo_sfx), "bytes")

# Verificar tipo pelo comando 'file'
resultado = subprocess.run(
    ["file", arquivo_sfx],
    capture_output=True,
    text=True
)

print("\nIdentificação:")
print(resultado.stdout)

Arquivo salvo: /content/TS251212.ex_
Tamanho: 326450 bytes

Identificação:
/content/TS251212.ex_: Zip archive data, at least v2.0 to extract, compression method=deflate



In [ ]:
# ============================================================
# TENTATIVA DE EXTRAÇÃO DO SFX
# ============================================================

pasta_saida = "/content/taxa_swap_2025_12_12"

if os.path.exists(pasta_saida):
    shutil.rmtree(pasta_saida)

os.makedirs(pasta_saida)

resultado = subprocess.run(
    [
        "7z",
        "x",
        arquivo_sfx,
        f"-o{pasta_saida}",
        "-y"
    ],
    capture_output=True,
    text=True
)

print("RETORNO DO 7-ZIP:")
print(resultado.stdout)
print(resultado.stderr)

print("\nARQUIVOS EXTRAÍDOS:")

for raiz, dirs, arquivos in os.walk(pasta_saida):
    for arquivo in arquivos:
        caminho = os.path.join(raiz, arquivo)
        print(caminho)

RETORNO DO 7-ZIP:

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.20GHz (406F0),ASM,AES-NI)

Scanning the drive for archives:
1 file, 326450 bytes (319 KiB)

Extracting archive: /content/TS251212.ex_
--
Path = /content/TS251212.ex_
Type = zip
Physical Size = 326450

Everything is Ok

Size:       468751
Compressed: 326450



ARQUIVOS EXTRAÍDOS:
/content/taxa_swap_2025_12_12/TS251212.ex_


GPT 3

In [ ]:
import zipfile
import io

# response ainda é o download original da B3

z = zipfile.ZipFile(io.BytesIO(response.content))

print("Número de entradas:", len(z.infolist()))
print()

for info in z.infolist():
    print("Nome:", repr(info.filename))
    print("Tamanho comprimido:", info.compress_size)
    print("Tamanho original:", info.file_size)
    print("Método:", info.compress_type)
    print("Offset:", info.header_offset)
    print("-" * 60)

Número de entradas: 1

Nome: 'TS251212.ex_'
Tamanho comprimido: 326312
Tamanho original: 468751
Método: 8
Offset: 0
------------------------------------------------------------


In [ ]:
# ============================================================
# INSPECIONAR O CONTEÚDO DA ÚNICA ENTRADA
# ============================================================

info = z.infolist()[0]

conteudo_interno = z.read(info)

print("Primeiros 100 bytes da entrada:")
print(conteudo_interno[:100])

print("\nTamanho:", len(conteudo_interno))

print("\nAssinatura:")
print(conteudo_interno[:4])

# Verificar se o conteúdo interno também é ZIP
print("\nÉ outro ZIP?", zipfile.is_zipfile(io.BytesIO(conteudo_interno)))

Primeiros 100 bytes da entrada:
b'MZ\xa1\x00\x02\x00\x00\x00 \x00\x00\x00\xff\xff\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00@\x00\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x18\x03\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'

Tamanho: 468751

Assinatura:
b'MZ\xa1\x00'

É outro ZIP? True


In [ ]:
# ============================================================
# TESTE 4 — abrir o ZIP interno do executável SFX
# ============================================================

import io
import zipfile

# conteudo_interno é o que extraímos da entrada do ZIP externo

zip_interno = zipfile.ZipFile(io.BytesIO(conteudo_interno))

print("Arquivos encontrados no ZIP interno:")
print("=" * 70)

for info in zip_interno.infolist():
    print(
        f"{info.filename:<40} "
        f"{info.file_size:>12,} bytes"
    )

Arquivos encontrados no ZIP interno:
TaxaSwap.txt                                2,283,048 bytes


In [ ]:
# ============================================================
# TESTE 5 — extrair TaxaSwap.txt
# ============================================================

nomes = zip_interno.namelist()

print("Arquivos:", nomes)

candidatos = [
    nome for nome in nomes
    if "taxaswap" in nome.lower()
]

if not candidatos:
    raise Exception(
        "Não encontrei TaxaSwap dentro do ZIP interno."
    )

nome_taxaswap = candidatos[0]

taxaswap_bytes = zip_interno.read(nome_taxaswap)

print("\nEncontrado:", nome_taxaswap)
print("Tamanho:", len(taxaswap_bytes), "bytes")

# Decodificar
taxaswap_texto = taxaswap_bytes.decode("latin-1")

print("\n" + "=" * 100)
print("PRIMEIRAS 20 LINHAS")
print("=" * 100)

for i, linha in enumerate(taxaswap_texto.splitlines()[:20], 1):
    print(f"{i:02d}: {repr(linha)}")

Arquivos: ['TaxaSwap.txt']

Encontrado: TaxaSwap.txt
Tamanho: 2283048 bytes

PRIMEIRAS 20 LINHAS
01: '0000010010120251212T1021  LFT            0008000052+00000000133000M00080'
02: '0000020010120251212T1021  LFT            0026300179-00000000143000M00263'
03: '0000030010120251212T1021  LFT            0044400300+00000000304000M00444'
04: '0000040010120251212T1021  LFT            0062800429+00000000410000M00628'
05: '0000050010120251212T1021  LFT            0081000553+00000000486000F00810'
06: '0000060010120251212T1021  LFT            0099400681+00000000561000M00994'
07: '0000070010120251212T1021  LFT            0117500801+00000000723000M01175'
08: '0000080010120251212T1021  LFT            0136100930+00000000893000M01361'
09: '0000090010120251212T1021  LFT            0154001052+00000000985000M01540'
10: '0000100010120251212T1021  LFT            0163401114+00000000999000M01634'
11: '0000110010120251212T1021  LFT            0172501178+00000001005000M01725'
12: '0000120010120251212T1021  LFT